## Reading 39-1 - One Kernel, Many Engines

By the end of this reading, you will learn:
<ol>
    <li>Why modern spacecraft computers contain several different kinds of processor</li>
    <li>How CPUs, GPUs, DSPs, and NPUs compute the same operation with different arithmetic</li>
    <li>Why the same calculation returns different answers on different engines, and how to validate each one</li>
    <li>How fixed-point overflow can be silent or visible, depending on a design choice</li>
</ol>

### Getting the Reading Code

<b>1.</b> Log into the Notre Dame Machines as you performed on Homework 1 using Tabby Terminal 

Open Tabby Terminal on your machine.

Type in: <code>ssh -Y username@student13.cse.nd.edu</code>
<ul><li>"username" is your NetID.</li>
<ul>
<li>Example: my email is matt.morrison@nd.edu, but my NetID is mmorri22, so I would enter:<p>
<code>ssh -Y mmorri22@student13.cse.nd.edu</code><p>
The <code>-Y</code> is included to treat the Notre Dame machines as a trusted client<p>
</li>
</ul>
</ul>

2. Next, we will create a directory where you will save your Reading Code. First, we will go into the Dropbox directory where you will submit your code:

<code>cd esc-courses</code>

<code>cd fa26-cse-20133.01</code>

<code>cd dropbox</code>

Make a directory for you to practice your code for this lecture

    mkdir reading39
    cd reading39

Enable script so you can run the most up-to-date versions of C and C++:

    scl enable gcc-toolset-15 bash

> <b>Note:</b> If you use the <code>-std=c2x</code> and get the following error, this means you did not run the enable command<br>
> <code>gcc: error: unrecognized command line option ‘-std=c2x’; did you mean ‘-std=c1x’?</code>

To obtain the code for today's lecture, perform the following commands:

    wget https://raw.githubusercontent.com/mmorri22/cse20133/main/readings/lec39/setup.sh
    chmod a+rx setup.sh
    ./setup.sh
    
To confirm, perform the <code>ls</code> command. You will see the following:

    Makefile  precision.c

### Heterogeneous Computers

The computers in your laptop and phone are no longer a single processor. Amalaye Oyake's talk describes a modern platform as several kinds of engine sharing one chip or one board:
<ul>
    <li>A <b>CPU</b> for control, scheduling, and decisions</li>
    <li>A <b>GPU</b> for massively parallel floating-point arithmetic</li>
    <li>An <b>NPU</b> (neural processing unit) for neural networks, using low-precision integers</li>
    <li>A <b>DSP</b> (digital signal processor) for filtering, often using fixed-point arithmetic</li>
</ul>

Spacecraft are following the same path. NASA's <b>Ingenuity</b> Mars helicopter flew a commercial Snapdragon processor running Linux for vision and navigation, alongside two small safety microcontrollers for flight control and an FPGA for sensor input. Ama draws the lesson as a pattern: <b>fast, complex engines do the heavy computing, and a small, verified core owns the safety loop.</b> Ingenuity's own history makes the point: its first flight on Mars was delayed after a <b>watchdog timer</b>, the safety mechanism from Lecture 37, expired while the computer was switching into flight mode.

### One Kernel - <a href = "https://github.com/mmorri22/cse20133/blob/main/readings/lec39/precision.c">precision.c</a>

In Lecture 33, you met GEMM, the matrix multiply that every one of these engines exists to accelerate. It shows up throughout a spacecraft: coordinate transformations, image processing, neural networks, and the Kalman filter from Lecture 38, whose multi-dimensional form multiplies matrices at every step.

<code>precision.c</code> computes one matrix-vector product, <code>y = A x</code>, the way four different engines would:

<table>
    <tr><th>Engine</th><th>Arithmetic</th><th>How a value is stored</th></tr>
    <tr><td>FP64</td><td>64-bit floating point, on a CPU</td><td>a <code>double</code>: the <b>reference</b></td></tr>
    <tr><td>FP32</td><td>32-bit floating point, common on GPUs</td><td>a <code>float</code></td></tr>
    <tr><td>Q15</td><td>16-bit fixed point, common on DSPs</td><td>an <code>int16_t</code> equal to the value times 32768</td></tr>
    <tr><td>INT8</td><td>8-bit integers, common on NPUs</td><td>an <code>int8_t</code> from -127 to 127, plus one scale factor for the whole array</td></tr>
</table>

    $ make precision
    y[0] from each engine:
      FP64 -0.066751611352
      FP32 -0.066751621664
      Q15  -0.066741943359
      INT8 -0.066065718530

Same matrix, same vector, same operation. <b>Four different answers.</b> None of the engines has a bug. Each one simply keeps a different number of bits, as you saw with IEEE 754 at the start of the semester and with machine epsilon in Lecture 33.

### Validate Every Engine Against a Reference

So which answer is correct? Ama's slide answers with a rule: <b>validate every accelerated kernel against a reference implementation, with a stated tolerance.</b> The tolerance is chosen in advance from what each engine can achieve and what the mission needs, and written down, as in Homework 9:

    engine    max error    tolerance
    FP32      1.644e-08      1.0e-06   PASS
    Q15       5.855e-05      1.0e-03   PASS
    INT8      9.927e-04      2.0e-02   PASS

The errors follow the number of bits: about 10<sup>-8</sup> for 32-bit floating point, 10<sup>-4</sup> for 16-bit fixed point, and 10<sup>-3</sup> for 8-bit integers. An INT8 error of 0.001 is fine for recognizing a crater in an image. It is not fine for computing a thruster burn. Choosing which engine runs which calculation is part of the design, and the tolerance is how you prove the choice was safe.

### Fixed Point Has to Be Scaled

Floating point adjusts its exponent automatically. Fixed point does not: a Q15 value must stay between -1 and 1. That is why <code>precision.c</code> divides every matrix entry by 16 before multiplying. What happens if you forget?

    Sixteen products of 0.9 * 0.9 (true sum 12.96):
      16-bit accumulator, wraps:     0.95947
      wide accumulator, saturates:   0.99997

Neither answer is 12.96, because 12.96 cannot exist in Q15. But look at the two wrong answers:
<ul>
    <li>The <b>wrapping</b> result, 0.95947, looks like a perfectly normal value. Nothing about it says that anything went wrong. It is the same silent failure as the <code>int16_t</code> telemetry field in Lecture 32 and the counters in Lecture 36.</li>
    <li>The <b>saturating</b> result is pinned at the largest possible value. It is still wrong, but it is wrong in a way software can <i>detect</i>: a value stuck at the maximum is a signal that the input was out of range.</li>
</ul>

This is why real DSPs provide wide accumulators and saturating arithmetic in hardware, and why <code>to_q15</code> in the example saturates instead of wrapping. When a value cannot be represented, fail visibly.

### <font color = "red">Class Introduction Question #1 - What is a heterogeneous computer? Using Ama's pattern, which engine on a spacecraft should run the safety loop, and why?</font>

### <font color = "red">Class Introduction Question #2 - <code>precision.c</code> computed the same product four ways and got four answers. Why, and how do you decide whether an engine's answer is acceptable?</font>

### <font color = "red">Class Introduction Question #3 - When the Q15 sum overflowed, why is the saturated result (0.99997) safer than the wrapped result (0.95947), even though both are wrong?</font>

### The next reading for this lecture is <a href = "https://github.com/mmorri22/cse20133/blob/main/readings/lec39/Reading%2039-2.ipynb">Reading 39-2 - Continuous Qualification, and Looking Back</a>